In [1]:
# Install Gemini API and Pydantic libraries
!pip install -q google-genai pydantic

In [3]:
#task 4.1

from google import genai
from getpass import getpass

# Enter API key securely at runtime
API_KEY = getpass("Paste your Gemini API key: ")

# Initialize Gemini client
client = genai.Client(api_key=API_KEY)

print("Gemini client initialized successfully!")

Paste your Gemini API key: ··········
Gemini client initialized successfully!


In [4]:
#define model and json schema

import json
from google.genai import types

# Gemini Flash model
MODEL_NAME = "gemini-2.5-flash"

# Define structured JSON response schema
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

print("Model and JSON schema configured successfully!")

Model and JSON schema configured successfully!


In [5]:
SYSTEM_TEXT = """
You are a healthcare appointment no-show analysis assistant.

Your task is to analyse call-centre notes and identify the most likely
reason why a patient missed or may miss an appointment.

Use only the information provided in the note.
Do not invent facts or make unsupported assumptions.
Return a structured response following the requested JSON schema.
Keep the summary within 25 words.
Confidence must be a number between 0 and 1.
"""

print("System instruction configured successfully!")

System instruction configured successfully!


In [6]:
#build the CRAFT prompt

def build_prompt(note_text):

    prompt = f"""
    CONTEXT:
    We are analysing patient call-centre notes to understand
    why appointments are missed.

    ROLE:
    Act as a healthcare appointment analysis assistant.

    ACTION:
    Analyse the following call-centre note and identify the
    most likely reason for the missed appointment.

    Allowed root_cause values:
    - forgot_or_no_reminder
    - distance_or_transport
    - cost_or_fee
    - illness_or_emergency
    - work_or_schedule
    - other

    TONE:
    Be objective, concise and evidence-based.
    Do not invent information or assume missing details.

    FORMAT:
    Return only valid JSON with these four fields:
    root_cause, confidence, summary, recommended_action.

    Rules:
    - confidence must be between 0 and 1.
    - summary must contain at most 25 words.
    - root_cause must use one of the allowed values.
    - recommended_action must be practical and relevant.

    CALL-CENTRE NOTE:
    {note_text}
    """

    return prompt

print("CRAFT prompt function created successfully!")

CRAFT prompt function created successfully!


In [7]:
# Import pandas
import pandas as pd

# Load follow-up notes
notes_df = pd.read_csv("/content/followup_notes.csv")

# Check dataset
print("Dataset Shape:", notes_df.shape)
print("\nColumns:", notes_df.columns.tolist())

# Display N001
print("\nN001 Note:")
print(notes_df[notes_df["note_id"] == "N001"].to_string(index=False))

Dataset Shape: (30, 3)

Columns: ['note_id', 'appointment_id', 'note_text']

N001 Note:
note_id appointment_id                                                                                                             note_text
   N001          A0872 Patient said she never got any reminder message and completely forgot about the visit. Wants to rebook for next week.


In [13]:
def analyse_note(note_text):

    # Build CRAFT prompt
    prompt = build_prompt(note_text)

    # Generate Gemini response (removed unsupported thinking_config)
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    # Parse JSON response
    data = json.loads(response.text)

    return data

print("analyse_note function updated successfully without thinking_config!")

analyse_note function updated successfully without thinking_config!


In [14]:
# Temporarily override MODEL_NAME to the supported gemini-3.8-flash
MODEL_NAME = "gemini-3.8-flash"

# Analyse N001
n001_result = analyse_note(n001_note)

# Print parsed dictionary
print(n001_result)

{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.95, 'summary': 'Patient missed appointment after not receiving a reminder and forgetting the visit; requested to reschedule for next week.', 'recommended_action': 'Verify patient contact details and notification preferences, then reschedule the appointment for next week.'}


In [15]:
# Verify response structure
required_fields = [
    "root_cause",
    "confidence",
    "summary",
    "recommended_action"
]

print("Returned fields:", list(n001_result.keys()))
print("All required fields present:",
      set(n001_result.keys()) == set(required_fields))

print("Root cause:", n001_result["root_cause"])
print("Confidence:", n001_result["confidence"])
print("Summary word count:", len(n001_result["summary"].split()))

Returned fields: ['root_cause', 'confidence', 'summary', 'recommended_action']
All required fields present: True
Root cause: forgot_or_no_reminder
Confidence: 0.95
Summary word count: 18


In [16]:
# Import Pydantic components-- task 4.2
from pydantic import BaseModel, Field, ValidationError

# Define allowed root causes
ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]

# Define structured validation model
class FollowupInsight(BaseModel):

    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

# Validate root cause against allowed values
def validate_root_cause(value):
    if value not in ALLOWED_ROOT_CAUSES:
        raise ValueError(f"Invalid root_cause: {value}")

print("Pydantic validation model created successfully!")

Pydantic validation model created successfully!


In [17]:
import time

# Store processed note results
processed_notes = []

# Process the first 10 notes
for _, row in notes_df.head(10).iterrows():

    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row["note_text"]

    # Default result structure
    result = {
        "note_id": note_id,
        "appointment_id": appointment_id,
        "root_cause": "",
        "confidence": "",
        "summary": "",
        "recommended_action": "",
        "validation_status": "",
        "raw_response": ""
    }

    # Check missing or blank notes first
    if pd.isna(note_text) or str(note_text).strip() == "":
        result.update({
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank"
        })

        print(f"{note_id}: Blank note skipped")

    else:
        try:
            # Call Gemini API
            data = analyse_note(str(note_text))

            # Preserve the original model response as JSON text
            result["raw_response"] = json.dumps(
                data, ensure_ascii=False
            )

            # Copy fields actually returned by the model
            for field in [
                "root_cause",
                "confidence",
                "summary",
                "recommended_action"
            ]:
                if field in data:
                    result[field] = data[field]

            # Validate using Pydantic
            insight = FollowupInsight(**data)

            # Validate allowed root cause
            validate_root_cause(insight.root_cause)

            result["validation_status"] = "valid"
            print(f"{note_id}: Valid")

        except Exception as e:
            result["validation_status"] = "invalid"
            print(f"{note_id}: Invalid - {e}")

        # Pause between API calls
        time.sleep(13)

    processed_notes.append(result)

print("\nProcessing completed!")
print("Total notes processed:", len(processed_notes))

N001: Valid
N002: Valid
N003: Valid
N004: Blank note skipped
N005: Valid
N006: Valid
N007: Valid
N008: Valid
N009: Valid
N010: Valid

Processing completed!
Total notes processed: 10


In [18]:
# Convert processed results into a DataFrame
notes_analysed_df = pd.DataFrame(processed_notes)

# Arrange columns in the required order
notes_analysed_df = notes_analysed_df[
    [
        "note_id",
        "appointment_id",
        "root_cause",
        "confidence",
        "summary",
        "recommended_action",
        "validation_status",
        "raw_response"
    ]
]

# Save the analysed notes
notes_analysed_df.to_csv(
    "notes_analysed.csv",
    index=False
)

# Verify exported file
print("File exported: notes_analysed.csv")
print("Shape:", notes_analysed_df.shape)

print("\nRoot Cause Distribution:")
print(notes_analysed_df["root_cause"].value_counts())

print("\nValidation Status:")
print(notes_analysed_df["validation_status"].value_counts())

File exported: notes_analysed.csv
Shape: (10, 8)

Root Cause Distribution:
root_cause
forgot_or_no_reminder    2
distance_or_transport    2
other                    2
work_or_schedule         2
cost_or_fee              1
illness_or_emergency     1
Name: count, dtype: int64

Validation Status:
validation_status
valid            9
skipped_blank    1
Name: count, dtype: int64


In [19]:
# Expected root causes from assignment
expected_causes = {
    "N001": "forgot_or_no_reminder",
    "N002": "distance_or_transport",
    "N003": "cost_or_fee",
    "N004": "other",
    "N005": "illness_or_emergency",
    "N006": "work_or_schedule",
    "N007": "forgot_or_no_reminder",
    "N008": "distance_or_transport",
    "N009": "other",
    "N010": "work_or_schedule"
}

# Compare actual and expected results
notes_analysed_df["expected_root_cause"] = (
    notes_analysed_df["note_id"].map(expected_causes)
)

notes_analysed_df["match"] = (
    notes_analysed_df["root_cause"]
    == notes_analysed_df["expected_root_cause"]
)

print(notes_analysed_df[
    ["note_id", "root_cause", "expected_root_cause", "match"]
].to_string(index=False))

# Exclude blank N004 from agreement calculation
non_blank = notes_analysed_df[
    notes_analysed_df["note_id"] != "N004"
]

print("\nCorrect matches:", non_blank["match"].sum(), "out of 9")

note_id            root_cause   expected_root_cause  match
   N001 forgot_or_no_reminder forgot_or_no_reminder   True
   N002 distance_or_transport distance_or_transport   True
   N003           cost_or_fee           cost_or_fee   True
   N004                 other                 other   True
   N005  illness_or_emergency  illness_or_emergency   True
   N006      work_or_schedule      work_or_schedule   True
   N007 forgot_or_no_reminder forgot_or_no_reminder   True
   N008 distance_or_transport distance_or_transport   True
   N009                 other                 other   True
   N010      work_or_schedule      work_or_schedule   True

Correct matches: 9 out of 9


In [20]:
#task 4.3

# Import Pydantic validation error
from pydantic import ValidationError

# Test Case 1: Invalid confidence value
bad_reply_1 = {
    "root_cause": "forgot_or_no_reminder",
    "confidence": 1.7,
    "summary": "Patient forgot the appointment",
    "recommended_action": "Send a reminder"
}

# Test Case 2: Missing recommended_action
bad_reply_2 = {
    "root_cause": "distance_or_transport",
    "confidence": 0.8,
    "summary": "Patient had transport issues"
}

# Validate first bad reply
try:
    FollowupInsight(**bad_reply_1)
except ValidationError as e:
    print("Test 1 - Invalid confidence:")
    print(e)

# Validate second bad reply
try:
    FollowupInsight(**bad_reply_2)
except ValidationError as e:
    print("\nTest 2 - Missing recommended_action:")
    print(e)

Test 1 - Invalid confidence:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Test 2 - Missing recommended_action:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'distance_...t had transport issues'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [21]:
# Define a simple V1 prompt
def build_v1_prompt(note_text):
    return f"Classify this note:\n{note_text}"


# Define a common Gemini call for prompt comparison
def test_prompt(prompt):

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    return json.loads(response.text)


# Extract N007 and N009
n007_note = notes_df.loc[
    notes_df["note_id"] == "N007", "note_text"
].iloc[0]

n009_note = notes_df.loc[
    notes_df["note_id"] == "N009", "note_text"
].iloc[0]

print("N007:", n007_note)
print("\nN009:", n009_note)

N007: Bhool gaya, SMS nahi aaya tha. Dobara booking karni hai.

N009: Phone was switched off, could not reach the patient after three attempts.


In [22]:
# Test V1 and V2 prompts on N007 and N009
comparison_results = []

test_cases = [
    ("N007", n007_note),
    ("N009", n009_note)
]

for note_id, note_text in test_cases:

    # V1: Simple prompt
    v1_data = test_prompt(build_v1_prompt(note_text))

    # V2: CRAFT prompt
    v2_data = test_prompt(build_prompt(note_text))

    # Check allowed root causes
    comparison_results.append({
        "Note": note_id,
        "Prompt": "V1",
        "Root Cause": v1_data.get("root_cause", ""),
        "Allowed": v1_data.get("root_cause") in ALLOWED_ROOT_CAUSES
    })

    comparison_results.append({
        "Note": note_id,
        "Prompt": "V2",
        "Root Cause": v2_data.get("root_cause", ""),
        "Allowed": v2_data.get("root_cause") in ALLOWED_ROOT_CAUSES
    })

    # Respect free-tier rate limit
    time.sleep(13)

# Display comparison table
comparison_df = pd.DataFrame(comparison_results)

print(comparison_df.to_string(index=False))

Note Prompt                                          Root Cause  Allowed
N007     V1 Did not receive SMS reminder and forgot appointment    False
N007     V2                               forgot_or_no_reminder     True
N009     V1 Patient unreachable due to phone being switched off    False
N009     V2                                               other     True


In [23]:
# Count API calls across Tasks 4.1 to 4.3

task_4_1_calls = 1
task_4_2_calls = 9
task_4_3_calls = 4

total_api_calls = (
    task_4_1_calls +
    task_4_2_calls +
    task_4_3_calls
)

print("Task 4.1 API Calls:", task_4_1_calls)
print("Task 4.2 API Calls:", task_4_2_calls)
print("Task 4.3 API Calls:", task_4_3_calls)

print("\nTotal API Calls:", total_api_calls)

Task 4.1 API Calls: 1
Task 4.2 API Calls: 9
Task 4.3 API Calls: 4

Total API Calls: 14


In [25]:
#task 4.4

# Load cleaned appointment dataset
appointments_df = pd.read_csv("/content/appointments_clean.csv")

# Merge GenAI notes with appointment data
merged_df = notes_analysed_df.merge(
    appointments_df,
    on="appointment_id",
    how="left"
)

# Verify merge
print("Appointments Shape:", appointments_df.shape)
print("Notes Shape:", notes_analysed_df.shape)
print("Merged Shape:", merged_df.shape)

print("\nMerged Dataset Preview:")
print(
    merged_df[
        ["note_id", "appointment_id", "root_cause", "reminder_sent"]
    ].to_string(index=False)
)

Appointments Shape: (990, 16)
Notes Shape: (10, 10)
Merged Shape: (10, 25)

Merged Dataset Preview:
note_id appointment_id            root_cause reminder_sent
   N001          A0872 forgot_or_no_reminder            No
   N002          A0897 distance_or_transport            No
   N003          A0526           cost_or_fee           Yes
   N004          A0926                 other            No
   N005          A0534  illness_or_emergency           Yes
   N006          A0507      work_or_schedule           Yes
   N007          A0333 forgot_or_no_reminder            No
   N008          A0222 distance_or_transport           Yes
   N009          A0977                 other           Yes
   N010          A0596      work_or_schedule            No


In [26]:
# Cross-tabulate root causes against reminder status
reminder_crosstab = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"]
)

print("Root Cause vs Reminder Status")
print(reminder_crosstab)

# Check total appointments with no reminder
print(
    "\nAppointments without reminder:",
    (merged_df["reminder_sent"] == "No").sum(),
    "out of",
    len(merged_df)
)

# Check forgotten appointments without reminder
forgotten = merged_df[
    merged_df["root_cause"] == "forgot_or_no_reminder"
]

print(
    "\nForgotten appointments without reminder:",
    (forgotten["reminder_sent"] == "No").sum(),
    "out of",
    len(forgotten)
)

Root Cause vs Reminder Status
reminder_sent          No  Yes
root_cause                    
cost_or_fee             0    1
distance_or_transport   1    1
forgot_or_no_reminder   2    0
illness_or_emergency    0    1
other                   1    1
work_or_schedule        1    1

Appointments without reminder: 5 out of 10

Forgotten appointments without reminder: 2 out of 2
